# Bài 2 — Vẽ đẹp với Annotators

**Mục tiêu:** Biến output khô khan thành hình ảnh trực quan chuyên nghiệp — và so sánh các kiểu vẽ ngay trên cửa sổ.

## 0. Chuẩn bị (asset video + display.py dùng chung)

In [2]:
!pip install -q supervision ultralytics "supervision[assets]"

In [3]:
from supervision.assets import download_assets, VideoAssets

download_assets(VideoAssets.VEHICLES)
print(VideoAssets.VEHICLES.value)  # "vehicles.mp4"

[2026-08-18 09:49:23] [INFO] supervision.assets.downloader - vehicles.mp4 asset download complete.
vehicles.mp4


In [4]:
%%writefile display.py
# display.py — hàm hiển thị dùng chung cho toàn giáo trình
import cv2

WINDOW_NAME = "Supervision - Live"
MAX_DISPLAY_WIDTH = 1280   # thu nhỏ frame cho vừa màn hình (chỉ để XEM, không ảnh hưởng xử lý)


def show_frame(frame, window_name: str = WINDOW_NAME, wait: int = 1) -> bool:
    """Hiện frame lên cửa sổ. Trả về False nếu người dùng bấm Q/ESC (muốn thoát).

    wait=1  -> dùng cho video (hiện liên tục, không chặn)
    wait=0  -> dùng cho ảnh tĩnh (dừng lại chờ bấm phím bất kỳ)
    """
    h, w = frame.shape[:2]
    if w > MAX_DISPLAY_WIDTH:                      # thu nhỏ để vừa màn hình
        scale = MAX_DISPLAY_WIDTH / w
        frame = cv2.resize(frame, (int(w * scale), int(h * scale)))

    cv2.imshow(window_name, frame)
    key = cv2.waitKey(wait) & 0xFF
    if key in (ord("q"), ord("Q"), 27):            # Q hoặc ESC -> thoát
        return False
    return True


def close_windows():
    cv2.destroyAllWindows()

Overwriting display.py


## 2.1. Cặp đôi cơ bản: Box + Label

In [5]:
import cv2
import supervision as sv
from ultralytics import YOLO

from display import show_frame, close_windows

model = YOLO("yolov8n.pt")
image = next(sv.get_video_frames_generator("vehicles.mp4"))
results = model(image)[0]
detections = sv.Detections.from_ultralytics(results)

# Khởi tạo annotator (tạo 1 lần, dùng lại nhiều lần)
box_annotator = sv.BoxAnnotator(thickness=2)
label_annotator = sv.LabelAnnotator(
    text_scale=0.5,
    text_thickness=1,
    text_position=sv.Position.TOP_LEFT,
)

# Tạo nhãn tùy biến: "car 0.87"
labels = [
    f"{class_name} {conf:.2f}"
    for class_name, conf
    in zip(detections.data["class_name"], detections.confidence)
]

# Vẽ (luôn copy để giữ ảnh gốc)
annotated = image.copy()
annotated = box_annotator.annotate(scene=annotated, detections=detections)
annotated = label_annotator.annotate(scene=annotated, detections=detections, labels=labels)

# 🖥️ Hiện lên cửa sổ xem ngay (bấm phím bất kỳ để đóng)
show_frame(annotated, wait=0)
close_windows()

# (Tùy chọn) muốn lưu lại thì thêm:
cv2.imwrite("bai2_output.jpg", annotated)


0: 384x640 3 cars, 1 truck, 106.2ms
Speed: 3.9ms preprocess, 106.2ms inference, 1.6ms postprocess per image at shape (1, 3, 384, 640)


True

## 2.2. Bộ sưu tập annotator — duyệt xem từng cái trên cửa sổ

Mỗi annotator hiện lên lần lượt — bấm phím bất kỳ để xem kiểu tiếp theo, Q để dừng.

In [6]:
annotators = {
    "RoundBox": sv.RoundBoxAnnotator(),          # Box bo góc — nhìn hiện đại
    "BoxCorner": sv.BoxCornerAnnotator(),        # Chỉ vẽ 4 góc — phong cách "quân sự"
    "Ellipse": sv.EllipseAnnotator(),            # Ellipse dưới chân — phân tích bóng đá
    "Circle": sv.CircleAnnotator(),              # Vòng tròn bao quanh
    "Dot": sv.DotAnnotator(),                    # Chấm tại tâm
    "Triangle": sv.TriangleAnnotator(),          # Tam giác trên đầu — kiểu game
    "Color": sv.ColorAnnotator(opacity=0.4),     # Tô màu cả vùng box
    "Blur": sv.BlurAnnotator(),                  # Làm mờ đối tượng (che biển số, mặt!)
    "Pixelate": sv.PixelateAnnotator(),          # Pixel hóa đối tượng
    # Cần model segmentation (yolov8n-seg.pt): MaskAnnotator, PolygonAnnotator, HaloAnnotator
    # Cần tracker_id (Bài 5): TraceAnnotator
    # HeatMapAnnotator: bản đồ nhiệt mật độ — hợp với video hơn ảnh tĩnh
}

for name, annotator in annotators.items():
    annotated = annotator.annotate(image.copy(), detections)
    cv2.putText(annotated, name, (20, 60),
                cv2.FONT_HERSHEY_SIMPLEX, 2, (0, 255, 255), 4)
    if not show_frame(annotated, window_name="So sanh Annotator", wait=0):
        break   # bấm Q thì dừng duyệt

close_windows()

## 2.3. Tùy biến màu sắc

In [7]:
# Bảng màu tùy chỉnh theo class
box_annotator = sv.BoxAnnotator(
    color=sv.ColorPalette.from_hex(["#ff0000", "#00ff00", "#0000ff", "#ffff00"]),
    color_lookup=sv.ColorLookup.CLASS,   # màu theo class
    # color_lookup=sv.ColorLookup.TRACK, # màu theo tracker_id (Bài 5)
)

annotated = box_annotator.annotate(image.copy(), detections)
show_frame(annotated, wait=0)
close_windows()

##  Checkpoint Bài 2

Bấm phím lần lượt duyệt hết bộ sưu tập annotator trên cửa sổ, mỗi kiểu có tên hiện góc trên.

**Bonus:** dùng `BlurAnnotator` che mờ toàn bộ người đi bộ và xem kết quả trên cửa sổ.